# GOLD: Table Analysis

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

## Grab Tables


In [0]:
aapl = spark.read.table("alpha_vantage.silver.aapl")
goog = spark.read.table("alpha_vantage.silver.goog")
inhd = spark.read.table("alpha_vantage.silver.inhd")
fgi = spark.read.table("alpha_vantage.silver.fgi")

In [0]:
def analysis(df):
    # Window ordered by trading date, partitioned per symbol
    w = Window.partitionBy("symbol").orderBy("date")

    gold_df = (
        df 
        .withColumn("close_5td_ago",  F.lag("close", 5).over(w))   # ~7 calendar days = 5 trading days
        .withColumn("close_21td_ago", F.lag("close", 21).over(w))  # ~30 calendar days
        .withColumn("close_63td_ago", F.lag("close", 63).over(w))  # ~90 calendar days

        .withColumn("price_change_5td",  F.col("close") - F.col("close_5td_ago"))
        .withColumn("price_change_21td", F.col("close") - F.col("close_21td_ago"))
        .withColumn("price_change_63td", F.col("close") - F.col("close_63td_ago"))

        .withColumn("pct_change_5td",  (F.col("close") - F.col("close_5td_ago"))  / F.col("close_5td_ago")  * 100)
        .withColumn("pct_change_21td", (F.col("close") - F.col("close_21td_ago")) / F.col("close_21td_ago") * 100)
        .withColumn("pct_change_63td", (F.col("close") - F.col("close_63td_ago")) / F.col("close_63td_ago") * 100)

        .withColumn("avg_volume_5td",  F.avg("volume").over(w.rowsBetween(-4, 0)))
        .withColumn("avg_volume_21td", F.avg("volume").over(w.rowsBetween(-20, 0)))
        .withColumn("avg_volume_63td", F.avg("volume").over(w.rowsBetween(-62, 0)))
    )
    return gold_df

In [0]:
aapl_gold = analysis(aapl)
fgi_gold = analysis(fgi)
goog_gold = analysis(goog)
inhd_gold = analysis(inhd)

In [0]:
goog_gold.display()

date,open,high,low,close,volume,symbol,close_5td_ago,close_21td_ago,close_63td_ago,price_change_5td,price_change_21td,price_change_63td,pct_change_5td,pct_change_21td,pct_change_63td,avg_volume_5td,avg_volume_21td,avg_volume_63td
2026-03-23,302.11,305.98,300.93,302.06,29326946,GOOGL,null,null,null,null,null,null,null,null,null,2.9326946E7,2.9326946E7,2.9326946E7
2026-03-24,299.2,299.92,290.33,290.44,36864278,GOOGL,null,null,null,null,null,null,null,null,null,3.3095612E7,3.3095612E7,3.3095612E7
2026-03-25,293.44,296.0,289.24,290.93,29460669,GOOGL,null,null,null,null,null,null,null,null,null,3.1883964333333332E7,3.1883964333333332E7,3.1883964333333332E7
2026-03-26,287.91,287.95,278.5,280.92,39080578,GOOGL,null,null,null,null,null,null,null,null,null,3.368311775E7,3.368311775E7,3.368311775E7
2026-03-27,277.275,279.37,273.95,274.34,35890612,GOOGL,null,null,null,null,null,null,null,null,null,3.41246166E7,3.41246166E7,3.41246166E7
2026-03-30,276.42,277.09,272.11,273.5,35141244,GOOGL,302.06,null,null,-28.560000000000002,null,null,-9.455075150632325,null,null,3.52874762E7,3.42940545E7,3.42940545E7
2026-03-31,278.04,288.08,277.09,287.56,43875400,GOOGL,290.44,null,null,-2.8799999999999955,null,null,-0.9915989533122144,null,null,3.66897006E7,3.566281814285714E7,3.566281814285714E7
2026-04-01,290.835,300.52,290.41,297.39,37684462,GOOGL,290.93,null,null,6.4599999999999795,null,null,2.2204654040490768,null,null,3.83344592E7,3.5915523625E7,3.5915523625E7
2026-04-02,290.69,298.08,289.45,295.77,21666465,GOOGL,280.92,null,null,14.849999999999966,null,null,5.286202477573674,null,null,3.48516366E7,3.433229488888889E7,3.433229488888889E7
2026-04-06,295.87,300.62,295.18,299.99,16945494,GOOGL,274.34,null,null,25.650000000000034,null,null,9.349712036159524,null,null,3.1062613E7,3.25936148E7,3.25936148E7


## Save Tables

In [0]:
aapl_gold.write.mode("overwrite").saveAsTable("alpha_vantage.gold.aapl_gold")
fgi_gold.write.mode("overwrite").saveAsTable("alpha_vantage.gold.fgi_gold")
goog_gold.write.mode("overwrite").saveAsTable("alpha_vantage.gold.goog_gold")
inhd_gold.write.mode("overwrite").saveAsTable("alpha_vantage.gold.inhd_gold")